In [2]:
%debug off
%define THREE_CLOCK(c1,c2,c3,c4,c5,c6,CB):(c1,c2,c3,c4,c5,c6,CB)
directive parameters [ k_clock = 0.005 ]
| c1 + c2 + CB ->{k_clock} c2 + c2+ CB
| c2 + c3+ CB ->{k_clock} c3 + c3+ CB
| c3 + c4 + CB->{k_clock} c4 + c4+ CB
| c4 + c5 + CB ->{k_clock} c5 + c5+ CB
| c5 + c6+ CB ->{k_clock} c6 + c6+ CB
| c6 + c1 + CB->{k_clock} c1 + c1+ CB
%end define
%define SUB_THREE_CLOCK(c1,c2,c3,c4,c5,c6,CB):(c1,c2,c3,c4,c5,c6,CB)
directive parameters [ k_clock = 0.035 ]
| c1 + c2 + CB ->{k_clock} c2 + c2+ CB
| c2 + c3+ CB ->{k_clock} c3 + c3+ CB
| c3 + c4 + CB->{k_clock} c4 + c4+ CB
| c4 + c5 + CB ->{k_clock} c5 + c5+ CB
| c5 + c6+ CB ->{k_clock} c6 + c6+ CB
| c6 + c1 + CB->{k_clock} c1 + c1+ CB
%end define
%define SUB_THREE_CLOCK2(c1,c2,c3,c4,c5,c6,CB):(c1,c2,c3,c4,c5,c6,CB)
directive parameters [ k_clock = 1 ]
| c1 + c2 + CB ->{k_clock} c2 + c2+ CB
| c2 + c3+ CB ->{k_clock} c3 + c3+ CB
| c3 + c4 + CB->{k_clock} c4 + c4+ CB
| c4 + c5 + CB ->{k_clock} c5 + c5+ CB
| c5 + c6+ CB ->{k_clock} c6 + c6+ CB
| c6 + c1 + CB->{k_clock} c1 + c1+ CB
%end define
%define ADD0C(A,B,CB):(Y,CB)
directive parameters [ k = 2;]
|CB+A->{k}CB+A+Y
|CB+B->{k}CB+B+Y
|CB+Y->{k}CB
%end define

%define ADDCC(A0,A1,B0,B1,C1,C2):(Y0,Y1,C1,C2)
//C1先执行，C2后执行,先加和再湮灭
directive parameters [ k = 2;]
%invoke ADD0C(A0,B0,C1):(Y0,C1)
%invoke ADD0C(A1,B1,C1):(Y1,C1)
|C2+Y0+Y1->{k}C2
%end define
%define ADDC(A0,A1,B0,B1,CB):(Y0,Y1,CB)
directive parameters [ k = 2;]
%invoke ADD0C(A0,B0,CB):(Y0,CB)
%invoke ADD0C(A1,B1,CB):(Y1,CB)
|CB+Y0+Y1->{k}CB
%end define

%define MUL0C(A,B,CB):(Y,CB)
directive parameters [ k = 2;]
|A+B+CB->{k}A+B+Y+CB
|Y+CB->{k}CB
%end define

%define MULCC(A0,A1,B0,B1,C1,C2):(Y0,Y1,C1,C2)
directive parameters [ k = 2;]
| 0 TMP1
| 0 TMP2
| 0 TMP3
| 0 TMP4
//一阶段计算没有遇到次序问题
%invoke MUL0C(A0,B0,C1):(TMP1,C1)
%invoke MUL0C(A1,B1,C1):(TMP2,C1)
%invoke MUL0C(A0,B1,C1):(TMP3,C1)
%invoke MUL0C(A1,B0,C1):(TMP4,C1)
%invoke ADD0C(TMP1,TMP2,C1):(Y0,C1)
%invoke ADD0C(TMP3,TMP4,C1):(Y1,C1)
|Y0+Y1+C2->{k}C2
%end define

%define MULC(A0,A1,B0,B1,CB):(Y0,Y1,CB)
//不需要内部时钟的版本
directive parameters [ k = 2;]
| 0 TMP1
| 0 TMP2
| 0 TMP3
| 0 TMP4
%invoke MUL0C(A0,B0,CB):(TMP1,CB)
%invoke MUL0C(A1,B1,CB):(TMP2,CB)
%invoke MUL0C(A0,B1,CB):(TMP3,CB)
%invoke MUL0C(A1,B0,CB):(TMP4,CB)
%invoke ADD0C(TMP1,TMP2,CB):(Y0,CB)
%invoke ADD0C(TMP3,TMP4,CB):(Y1,CB)
%end define


%define MACC(A0,W0,A1,W1,B0,W2,B1,W3,CB):(Y0,Y1,CB)
directive parameters [ k = 2;]

| 0 Y3
| 0 Y4
| 0 Y5
| 0 Y6
%invoke MULC(A0,A1,W0,W1,CB):(Y3,Y4,CB)
%invoke MULC(B0,B1,W2,W3,CB):(Y5,Y6,CB)
%invoke ADDC(Y3,Y4,Y5,Y6,CB):(Y0,Y1,CB)
%end define

%define TANHC(N0,N1,CB):(H0,H1,CB)
directive parameters [ k = 2]
| 1 c1
| 1e-6 c2
| 1e-6 c3
| 1e-6 c4
| 1e-6 c5
| 1 c6

%invoke SUB_THREE_CLOCK2(c1,c2,c3,c4,c5,c6,CB):(c1,c2,c3,c4,c5,c6,CB)

| c3+N0+N1+CB->{k}c3+CB
//正分量逻辑 (对应 dH0/dt = k*N0*(1 - H0^2))
|c5+N0+CB->{k}c5+N0+H0+CB
|N0+H0+H0+c5+CB->{k}c5+N0+H0+CB
|N0+c5+CB->{k}c5+CB
//负分量逻辑 (对应 dH1/dt = k*N1*(1 - H1^2))
|N1+c5+CB->{k}N1+H1+c5+CB
|N1+H1+H1+c5+CB->{k}N1+H1+c5+CB
|N1+c5+CB->{k}c5+CB

%end define

%define NEURON(X0P,X0N,W0P,W0N,X1P,X1N,W1P,W1N,WP,WN,CB):(YP,YN,CB)
//需要三时钟
| 1 c1
| 1e-6 c2
| 1e-6 c3
| 1e-6 c4
| 1e-6 c5
| 1 c6
| 0 YTMP
| 0 Y0
| 0 Y1
| 0 YTMP0
| 0 YTMP1

%invoke SUB_THREE_CLOCK(c1,c2,c3,c4,c5,c6,CB):(c1,c2,c3,c4,c5,c6,CB)
//第一步，MAC计算
%invoke MACC(X0P,W0P,X0N,W0N,X1P,W1P,X1N,W1N,c1):(Y0,Y1,c1)
//第二步 WP偏置计算
%invoke ADD0C(Y0,WP,c3):(YTMP0,c3)
%invoke ADD0C(Y1,WN,c3):(YTMP1,c3)
//第二步tanh计算
%invoke TANHC(YTMP0,YTMP1,c5):(YP,YN,c5)
%end define


Material state reset (macros and settings preserved).
Debug output enabled.
// Full Expanded Code:
directive simulation { 
  final=2000; 
  plots=[res1;res2;res3];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res1 = [c0]-[c1];
  res2 = [c2]-[c3];
  res3 = [c4]-[c5];
]	

| 4 AP
| 5 AN
| 0 BP
| 0 BN
| 0 c0
| 0 c1
| 0 c2
| 0 c3
| 0 c4
| 0 c5
| 1 CB
%invoke TEST_FUNC(AP,AN,CB):(c0,c1,CB)
%invoke NOISEC(AP,AN,CB):(BP,BN,CB)
%invoke TEST_FUNC(BP,BN,CB):(c2,c3,CB)
%invoke GET_STEPC(c0,c1,c2,c3,CB):(c4,c5,CB)




Debug output disabled.
Macro definitions registered: 12
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.


# 直接使用训练后权重

In [3]:
%reset
%export expanded
%debug on
%title "XOR Network (Trained Weights)"
directive simulation { 
  final=5000; 
  plots=[res;res1;res2];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res = [c0]-[c1];
  res1 = [H0P]-[H0N];
  res2 = [H1P] - [H1N]
]	

// === 输入信号 (测试用例: 可修改) ===
| 1 X1P | 0 X1N    // X1 = -1
| 0 X2P | 1 X2N    // X2 = -1
| 1 CB             // 偏置信号 = -1

// === 训练后权重 (双轨表示) ===

// --- N0 权重 ---
| 2.94304066 W00P | 0 W00N      // W00 = -2.943 → 贡献 +2.943
| 0 W01P | 2.73212848 W01N      // W01 = -2.732
| 0 W02P | 3.90841656 W02N      // W02 = -3.908

// --- N1 权重 ---
| 0 W10P | 2.62188713 W10N      // W10 = +2.622 → 贡献 -2.622
| 0 W11P | 3.38384533 W11N      // W11 = -3.384
| 0 W12P | 2.60482672 W12N      // W12 = -2.605

// --- N2 权重 ---
| 0 W20P | 3.77539904 W20N      // W20 = +3.775 → 贡献 -3.775
| 4.05530305 W21P | 0 W21N      // W21 = +4.055
| 0 W22P | 3.68841413 W22N      // W22 = -3.688

// === 中间变量 ===
| 0 H0P | 0 H0N
| 0 H1P | 0 H1N
| 0 c0  | 0 c1

// === 时钟 ===
| 1 clock1 | 1e-6 clock2 | 1e-6 clock3 | 1e-6 clock4 | 1e-6 clock5 | 1 clock6

%invoke THREE_CLOCK(clock1,clock2,clock3,clock4,clock5,clock6,CB):(clock1,clock2,clock3,clock4,clock5,clock6,CB)

// === 神经元调用 ===
%invoke NEURON(X1P, X1N, W01P, W01N, X2P, X2N, W02P, W02N, W00P, W00N, clock1):(H0P, H0N, clock1)
%invoke NEURON(X1P, X1N, W11P, W11N, X2P, X2N, W12P, W12N, W10P, W10N, clock1):(H1P, H1N, clock1)
%invoke NEURON(H0P, H0N, W21P, W21N, H1P, H1N, W22P, W22N, W20P, W20N, clock3):(c0, c1, clock3)

Material state reset (macros and settings preserved).
Debug output enabled.
// Full Expanded Code:
directive simulation { 
  final=2000; 
  plots=[res1;res2;res3];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res1 = [c0]-[c1];
  res2 = [c2]-[c3];
  res3 = [c4]-[c5];
]	

| 4 AP
| 5 AN
| 0 BP
| 0 BN
| 0 c0
| 0 c1
| 0 c2
| 0 c3
| 0 c4
| 0 c5
| 1 CB
%invoke TEST_FUNC(AP,AN,CB):(c0,c1,CB)
%invoke NOISEC(AP,AN,CB):(BP,BN,CB)
%invoke TEST_FUNC(BP,BN,CB):(c2,c3,CB)
%invoke GET_STEPC(c0,c1,c2,c3,CB):(c4,c5,CB)




Debug output disabled.
Macro definitions registered: 12
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.
Material state reset (macros and settings preserved).
Debug output enabled.
[DEBUG processCode] code.Length = 1480
[DEBUG processCode] codeWithoutDefs.Length = 1480
[DEBUG processCode] fullExpandedCode.Length = 32903
[DEBUG

### Simulation Results

**Simulation Time:** 0.00 to 5000.00
**Number of Time Points:** 1001 (requested: 1000)

#### Final Concentrations

| Species | Concentration |
|---------|---------------|
| [res1] | 0.983244 |
| [res2] | -0.982065 |
| [res] | 0.989473 |



Simulation completed successfully.

Time range: 0.00 to 5000.00
Number of time points: 1001 (requested: 1000)

Number of species: 3

Final concentrations:
  [res1]: 0.983244
  [res2]: -0.982065
  [res]: 0.989473


# 封装前馈网络

In [3]:
%define XOR_NETWORK(X1P,X1N,X2P,X2N,W01P,W01N,W02P,W02N,W00P,W00N,W11P,W11N,W12P,W12N,W10P,W10N,W21P,W21N,W22P,W22N,W20P,W20N,CB):(c0,c1,CB)

// === 时钟 ===
| 1 clock1 | 1e-6 clock2 | 1e-6 clock3 | 1e-6 clock4 | 1e-6 clock5 | 1 clock6

%invoke THREE_CLOCK(clock1,clock2,clock3,clock4,clock5,clock6,CB):(clock1,clock2,clock3,clock4,clock5,clock6,CB)

// === 神经元调用 ===
%invoke NEURON(X1P, X1N, W01P, W01N, X2P, X2N, W02P, W02N, W00P, W00N, clock1):(H0P, H0N, clock1)
%invoke NEURON(X1P, X1N, W11P, W11N, X2P, X2N, W12P, W12N, W10P, W10N, clock1):(H1P, H1N, clock1)
%invoke NEURON(H0P, H0N, W21P, W21N, H1P, H1N, W22P, W22N, W20P, W20N, clock3):(c0, c1, clock3)

%end define

Material state reset (macros and settings preserved).
Debug output enabled.
// Full Expanded Code:
directive simulation { 
  final=2000; 
  plots=[res1;res2;res3];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res1 = [c0]-[c1];
  res2 = [c2]-[c3];
  res3 = [c4]-[c5];
]	

| 4 AP
| 5 AN
| 0 BP
| 0 BN
| 0 c0
| 0 c1
| 0 c2
| 0 c3
| 0 c4
| 0 c5
| 1 CB
%invoke TEST_FUNC(AP,AN,CB):(c0,c1,CB)
%invoke NOISEC(AP,AN,CB):(BP,BN,CB)
%invoke TEST_FUNC(BP,BN,CB):(c2,c3,CB)
%invoke GET_STEPC(c0,c1,c2,c3,CB):(c4,c5,CB)




Debug output disabled.
Macro definitions registered: 12
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.
Macro definitions registered: 13
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.


In [5]:
%reset
%export expanded
%debug on
%title "XOR Network (Trained Weights)"
directive simulation { 
  final=5000; 
  plots=[res;res1;res2];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res = [c0]-[c1];
  res1 = [H0P]-[H0N];
  res2 = [H1P] - [H1N]
]	

// === 输入信号 (测试用例: 可修改) ===
| 1 X1P | 0 X1N    // X1 = -1
| 0 X2P | 1 X2N    // X2 = -1
| 1 CB             // 偏置信号 = -1

// === 训练后权重 (双轨表示) ===

// --- N0 权重 ---
| 2.94304066 W00P | 0 W00N      // W00 = -2.943 → 贡献 +2.943
| 0 W01P | 2.73212848 W01N      // W01 = -2.732
| 0 W02P | 3.90841656 W02N      // W02 = -3.908

// --- N1 权重 ---
| 0 W10P | 2.62188713 W10N      // W10 = +2.622 → 贡献 -2.622
| 0 W11P | 3.38384533 W11N      // W11 = -3.384
| 0 W12P | 2.60482672 W12N      // W12 = -2.605

// --- N2 权重 ---
| 0 W20P | 3.77539904 W20N      // W20 = +3.775 → 贡献 -3.775
| 4.05530305 W21P | 0 W21N      // W21 = +4.055
| 0 W22P | 3.68841413 W22N      // W22 = -3.688

// === 中间变量 ===
| 0 H0P | 0 H0N
| 0 H1P | 0 H1N
| 0 c0  | 0 c1

%invoke XOR_NETWORK(X1P,X1N,X2P,X2N,W01P,W01N,W02P,W02N,W00P,W00N,W11P,W11N,W12P,W12N,W10P,W10N,W21P,W21N,W22P,W22N,W20P,W20N,CB):(c0,c1,CB) as net1

[DEBUG processCode] code.Length = 672
[DEBUG processCode] codeWithoutDefs.Length = 1
[DEBUG processCode] fullExpandedCode.Length = 1
[DEBUG processCode] registry.Instances.Count = 73
[DEBUG processCode] registry.Definitions.Count = 13
Macro definitions registered: 13
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.
Material state reset (macros and settings preserved).
Debug output enabled.
[DEBUG processCode] code.Length = 1113
[DEBUG processCode] codeWithoutDefs.Length = 1113
[DEBUG processCode] fullExpandedCode.Length = 52115
[DEBUG processCode] registry.Instances.Count = 147
[DEBUG processCode] registry.Definitions.Count = 13
// Registered Macros:
//   THREE_CLOCK(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   SUB_THREE_CLOCK(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   SUB_THREE_CLOCK2(c1, c2, 

### Simulation Results

**Simulation Time:** 0.00 to 5000.00
**Number of Time Points:** 1001 (requested: 1000)

#### Final Concentrations

| Species | Concentration |
|---------|---------------|
| [res1] | 0 |
| [res2] | 0 |
| [res] | 0.989473 |



Simulation completed successfully.

Time range: 0.00 to 5000.00
Number of time points: 1001 (requested: 1000)

Number of species: 3

Final concentrations:
  [res1]: 0
  [res2]: 0
  [res]: 0.989473


# 双轨除法？
原文中未使用双轨除法，而是$\frac{\alpha}{\Delta W_i}$

双轨除法比较复杂，假设$A=A_1-A_0,B=B_1-B_0$，那么$\frac{A}{B}=\frac{A_1-A_0}{B_1-B_0}$

即使$A,B$已经是充分湮灭的，其求法还是得分为四个部分

$$
\begin{aligned}
都只剩正数项\\
\frac{A_0}{B_0} \rightarrow Y_0\\
都只剩负数项\\
\frac{A_1}{B_1} \rightarrow Y_0\\
其它\\
\frac{A_0}{B_1} \rightarrow Y_1\\
\frac{A_1}{B_0} \rightarrow Y_1\\
\end{aligned}
$$

In [4]:
%define DIV0C(A,B):(Y)
directive parameters [ k = 2;]
| A->{k} A + Y
| Y+B->{k}B
%end define

Material state reset (macros and settings preserved).
Debug output enabled.
// Full Expanded Code:
directive simulation { 
  final=2000; 
  plots=[res1;res2;res3];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res1 = [c0]-[c1];
  res2 = [c2]-[c3];
  res3 = [c4]-[c5];
]	

| 4 AP
| 5 AN
| 0 BP
| 0 BN
| 0 c0
| 0 c1
| 0 c2
| 0 c3
| 0 c4
| 0 c5
| 1 CB
%invoke TEST_FUNC(AP,AN,CB):(c0,c1,CB)
%invoke NOISEC(AP,AN,CB):(BP,BN,CB)
%invoke TEST_FUNC(BP,BN,CB):(c2,c3,CB)
%invoke GET_STEPC(c0,c1,c2,c3,CB):(c4,c5,CB)




Debug output disabled.
Macro definitions registered: 12
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.
Macro definitions registered: 13
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.
Macro definitions registered: 14
No 

论文中采用影子网络的并行方法在实现上似乎更加方便，其时钟周期简洁明了，对于无法使用`FOR`等循环步骤的带宏的CRN而言更加方便

具体的训练步骤包括：

1.主网络以及影子网络并行进行计算

2.主网络和影子网络输出值求损失函数

3.上一步结果做差，求得更新步长

4.同步更新主网络以及各个影子网络的步长

5.清除中间产物，恢复网络可用状态

# 求损失函数

$$
L=(Y-T)^2
$$


In [23]:
%define SUB0C(AP,AN,BP,BN,CB):(CP,CN,CB)
%invoke ADD0C(AP,BN,CB):(CP,CB)
%invoke ADD0C(AN,BP,CB):(CN,CB)
%end define

%define SQUAREC(AP,AN,CB):(CP,CN,CB)
%invoke MULC(AP,AN,AP,AN,CB):(CP,CN,CB)
%end define
%define COPY(A):(B)
directive parameters [ k = 2;]
|A->{k}A+B
|B->{k}
%end define
%define LOSSC(YP,YN,TP,TN,CB):(LP,LN,CB)
directive parameters [ k = 2;]
| 0 T0
| 0 T1
%invoke SUB0C(YP,YN,TP,TN,CB):(T0,T1,CB)
%invoke SQUAREC(T0,T1,CB):(LP,LN,CB)
%end define
%define NOISEC(YP,YN,CB):(CP,CN,CB)
| 0.1 WP
%invoke ADD0C(WP,YP,CB):(CP,CB)
%invoke COPY(YN):(CN)
%end define

%define GET_STEPC(YP,YN,TP,TN,CB):(WP,WN,CB)
| 0 W0
| 0 W1
| 10 W2
| 0 W3
%invoke SUB0C(YP,YN,TP,TN,CB):(W0,W1,CB)
%invoke MULC(W0,W1,W2,W3,CB):(WP,WN,CB)
%end define


[DEBUG processCode] code.Length = 746
[DEBUG processCode] codeWithoutDefs.Length = 4
[DEBUG processCode] fullExpandedCode.Length = 4
[DEBUG processCode] registry.Instances.Count = 448
[DEBUG processCode] registry.Definitions.Count = 21
Macro definitions registered: 21
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.


`LOSSC`用来求损失函数

`NOISEC`用来产生扰动值

`GET_STEPC`用来得到梯度

# 求梯度测试

目标函数$$y=x^2 + 2x$$

In [24]:
%define TEST_FUNC(IP,IN,CB):(OP,ON,CB)
| 2 W0
| 0 W1
| 0 O1P
| 0 O1N
| 0 O2P
| 0 O2N
%invoke SQUAREC(IP,IN,CB):(O1P,O1N,CB)
%invoke MULC(W0,W1,IP,IN,CB):(O2P,O2N,CB)
%invoke ADD0C(O1P,O2P,CB):(OP,CB)
%invoke ADD0C(O1N,O2N,CB):(ON,CB)
%end define

[DEBUG processCode] code.Length = 746
[DEBUG processCode] codeWithoutDefs.Length = 4
[DEBUG processCode] fullExpandedCode.Length = 4
[DEBUG processCode] registry.Instances.Count = 448
[DEBUG processCode] registry.Definitions.Count = 21
Macro definitions registered: 21
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.
[DEBUG processCode] code.Length = 247
[DEBUG processCode] codeWithoutDefs.Length = 1
[DEBUG processCode] fullExpandedCode.Length = 1
[DEBUG processCode] registry.Instances.Count = 429
[DEBUG processCode] registry.Definitions.Count = 21
Macro definitions registered: 21
No simulation performed (macro definitions only)

Note: Macro definitions are stored but not executed.
   Use %invoke to call macros in subsequent cells.


In [ ]:
%reset
%debug off
%title "TEST STEP"
directive simulation { 
  final=10; 
  plots=[res];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  res = [c0]-[c1];
]	

| 4 AP
| 5 AN
| 0 c0
| 0 c1
| 1 CB
%invoke TEST_FUNC(AP,AN,CB):(c0,c1,CB)

Material state reset (macros and settings preserved).
Debug output disabled.


### Simulation Results

**Simulation Time:** 0.00 to 10.00
**Number of Time Points:** 1001 (requested: 1000)

#### Final Concentrations

| Species | Concentration |
|---------|---------------|
| [res] | -1 |



Simulation completed successfully.

Time range: 0.00 to 10.00
Number of time points: 1001 (requested: 1000)

Number of species: 1

Final concentrations:
  [res]: -1


: 

$$y'=2x + 2$$

In [32]:
%reset
%export expanded
%debug on
%title "TEST STEP"
directive simulation { 
  final=10; 
  plots=[tmp0;tmp1;res1;res2;res3];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  tmp0 = [AP]-[AN];
  tmp1 = [BP]-[BN];
  res1 = [c0]-[c1];
  res2 = [c2]-[c3];
  res3 = [c4]-[c5];
]	

| 10 AP
| 0 AN
| 0 BP
| 0 BN
| 0 c0
| 0 c1
| 0 c2
| 0 c3
| 0 c4
| 0 c5
| 1 CB
%invoke TEST_FUNC(AP,AN,CB):(c0,c1,CB)
%invoke NOISEC(AP,AN,CB):(BP,BN,CB)
%invoke TEST_FUNC(BP,BN,CB):(c2,c3,CB)
%invoke GET_STEPC(c2,c3,c0,c1,CB):(c4,c5,CB)


Material state reset (macros and settings preserved).
Debug output enabled.
[DEBUG processCode] code.Length = 515
[DEBUG processCode] codeWithoutDefs.Length = 515
[DEBUG processCode] fullExpandedCode.Length = 13179
[DEBUG processCode] registry.Instances.Count = 765
[DEBUG processCode] registry.Definitions.Count = 21
// Registered Macros:
//   THREE_CLOCK(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   SUB_THREE_CLOCK(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   SUB_THREE_CLOCK2(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   ADD0C(A, B, CB) :(Y, CB) [rate params: ]
//     OK
//   ADDCC(A0, A1, B0, B1, C1, C2) :(Y0, Y1, C1, C2) [rate params: ]
//     OK
//   ADDC(A0, A1, B0, B1, CB) :(Y0, Y1, CB) [rate params: ]
//     OK
//   MUL0C(A, B, CB) :(Y, CB) [rate params: ]
//     OK
//   MULCC(A0, A1, B0, B1, C1, C2) :(Y0, Y1, C1, C2) [rate params: ]
//     OK
//   MUL

### Simulation Results

**Simulation Time:** 0.00 to 10.00
**Number of Time Points:** 1001 (requested: 1000)

#### Final Concentrations

| Species | Concentration |
|---------|---------------|
| [res1] | 120 |
| [res2] | 122.209 |
| [res3] | 21.7314 |
| [tmp0] | 10 |
| [tmp1] | 10.1 |



Simulation completed successfully.

Time range: 0.00 to 10.00
Number of time points: 1001 (requested: 1000)

Number of species: 5

Final concentrations:
  [res1]: 120
  [res2]: 122.209
  [res3]: 21.7314
  [tmp0]: 10
  [tmp1]: 10.1


似乎会产生大概0.1的误差，但整体趋势没有问题

下一步是需要实现权重更新机制，然后找一个简单的样例来进行测试

权重更新机制需要用到之前的宏实例、内部物质导出功能

样例可以是考虑对于刚刚定义的`TEST_FUNC`,设置一个权重，然后是两个权重，看看能不能最终收敛到正确的函数

那么目标函数设定为$$y=3x^2+2x$$

找上$10$个点吧



### DEEPSEEK生成的数据点（共11个）
| x   | y (真实值) | y (含噪声) |
|-----|-----------|-----------|
| -5  | 65        | 66.5      |
| -4  | 40        | 39.2      |
| -3  | 21        | 20.8      |
| -2  | 8         | 7.5       |
| -1  | 1         | 1.2       |
| 0   | 0         | -0.3      |
| 1   | 5         | 5.1       |
| 2   | 16        | 15.8      |
| 3   | 33        | 33.4      |
| 4   | 56        | 56.7      |
| 5   | 85        | 84.6      |



In [ ]:
%define TEST_FUNC1(IP,IN,CB):(OP,ON,CB)
| -1.5 W0
| 0 W1
| 3.2 W2
| 0 W3
| 0 O1P
| 0 O1N
| 0 O2P
| 0 O2N
| 0 O3P
| 0 O3N
%invoke SQUAREC(IP,IN,CB):(O1P,O1N,CB)
%invoke MULC(W2,W3,O1P,O1N,CB):(O3P,O3N,CB)
%invoke MULC(W0,W1,IP,IN,CB):(O2P,O2N,CB)
%invoke ADD0C(O1P,O3P,CB):(OP,CB)
%invoke ADD0C(O1N,O3N,CB):(ON,CB)
%end define

In [ ]:
%reset
%export expanded
%debug on
%title "TEST STEP"
directive simulation { 
  final=10; 
  plots=[tmp0;tmp1;res1;res2;res3];
}
directive simulator deterministic
directive parameters [ k = 0.003; u = 0.1 ]

directive rates [
  tmp0 = [AP]-[AN];
  tmp1 = [BP]-[BN];
  res1 = [c0]-[c1];
  res2 = [c2]-[c3];
  res3 = [c4]-[c5];
]	

| 10 AP
| 0 AN
| 0 BP
| 0 BN
| 0 c0
| 0 c1
| 0 c2
| 0 c3
| 0 c4
| 0 c5
| 1 CB
%invoke TEST_FUNC(AP,AN,CB):(c0,c1,CB)
%invoke NOISEC(AP,AN,CB):(BP,BN,CB)
%invoke TEST_FUNC(BP,BN,CB):(c2,c3,CB)
%invoke GET_STEPC(c2,c3,c0,c1,CB):(c4,c5,CB)


In [11]:
%export macro SQUAREC
%define SQUAREC(AP,AN,CB):(CP,CN,CB)
%invoke MULC(AP,AN,AP,AN,CB):(CP,CN,CB)
%end define

Material state reset (macros and settings preserved).
Debug output enabled.
[DEBUG processCode] code.Length = 466
[DEBUG processCode] codeWithoutDefs.Length = 466
[DEBUG processCode] fullExpandedCode.Length = 12749
[DEBUG processCode] registry.Instances.Count = 214
[DEBUG processCode] registry.Definitions.Count = 20
// Registered Macros:
//   THREE_CLOCK(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   SUB_THREE_CLOCK(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   SUB_THREE_CLOCK2(c1, c2, c3, c4, c5, c6, CB) :(c1, c2, c3, c4, c5, c6, CB) [rate params: ]
//     OK
//   ADD0C(A, B, CB) :(Y, CB) [rate params: ]
//     OK
//   ADDCC(A0, A1, B0, B1, C1, C2) :(Y0, Y1, C1, C2) [rate params: ]
//     OK
//   ADDC(A0, A1, B0, B1, CB) :(Y0, Y1, CB) [rate params: ]
//     OK
//   MUL0C(A, B, CB) :(Y, CB) [rate params: ]
//     OK
//   MULCC(A0, A1, B0, B1, C1, C2) :(Y0, Y1, C1, C2) [rate params: ]
//     OK
//   MUL